# [Sensitivity Analysis] The Routing Bottleneck (KAN-TabNet) | Step LR | Poker Hand

**Optimized Reference Configuration:** `n_d=20, n_a=20`, `kan_grid_size=5`, `kan_spline_order=3`, `initial_lr=0.002`

### Methodological Context: The Control Variables
To precisely isolate the effects of the network's internal routing capacity, this sensitivity analysis uses the exact same `StepLR` thermodynamic schedule, initial learning rate (`0.002`), cubic spline geometry (`k=3`), and grid resolution (`G=5`) as our optimized reference configuration.

By fixing the mathematical properties of the KAN layers and the optimization environment, we strictly guarantee that any performance variations observed in this notebook are purely the result of constricting the pipeline dimensions ($n_d$ and $n_a$).

### The Hypothesis
In this study, we evaluate a constricted routing pipeline by reducing both the decision and attention dimensions to `n_d=10, n_a=10`.

The Forest Cover dataset consists of 54 distinct features, the vast majority of which are highly sparse, one-hot encoded categorical arrays (such as 40 binary columns simply to represent soil type). Our reference configuration utilized a massive `20x20` routing matrix to process this wide array of inputs. We hypothesize that unlike lower-dimensional datasets (such as Poker Hand with 10 features), high-dimensional mixed data mathematically requires a massive internal routing matrix to successfully isolate and pass forward the relevant sparse 0s and 1s. This notebook serves to empirically evaluate if halving the pipeline to $10 \times 10$ creates a critical "Routing Bottleneck," causing the network to lose vital spatial information and artificially capping its topographical accuracy.

### Setup

In [1]:
%%capture
# install TabNet fork which allows switching between vanilla TabNet and KAN-TabNet
!pip install git+https://github.com/chuo-v/tabnet.git@v1.0.1-kan

In [2]:
import os
import json
import numpy as np
import random
import torch

seed = 11
random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

### Dataset Fetching

In [4]:
from sklearn.datasets import fetch_covtype
from sklearn.model_selection import train_test_split
from pytorch_tabnet.tab_model import TabNetClassifier

dataset = fetch_covtype()

X = dataset.data
# CovType is 1-indexed (1 to 7); PyTorch expects 0-indexed labels
y = dataset.target - 1

# divide dataset into 80% train, 20% temp (validation + test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.20, random_state=seed, stratify=y
)

# divide temp into 10% validation and 10% test
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=seed, stratify=y_temp
)

print(f"Train shape: {X_train.shape}")
print(f"Valid shape: {X_valid.shape}")
print(f"Test shape:  {X_test.shape}")

Train shape: (464809, 54)
Valid shape: (58101, 54)
Test shape:  (58102, 54)


### Model Training

In [5]:
MAX_EPOCHS = 1000

In [6]:
# Hyperparameters from original paper.
# Notes:
# - momentum is 1 - 0.7 (paper m_B) to align with PyTorch's BatchNorm API.
# - scheduler step_size of 18 epochs approximates the paper's 500 iterations
#   (based on a batch size of 16384 and ~464k training samples).
paper_config = {
    'n_steps': 5,
    'gamma': 1.5,
    'n_independent': 2,
    'n_shared': 2,
    'lambda_sparse': 1e-4,
    'momentum': 0.3,
    'optimizer_fn': torch.optim.Adam,
    'scheduler_fn': torch.optim.lr_scheduler.StepLR,
    'scheduler_params': dict(step_size=18, gamma=0.95),
    'mask_type': 'sparsemax'
}

clf_kan = TabNetClassifier(
    n_d=10,
    n_a=10,
    **paper_config,
    clip_value=2.,
    optimizer_params=dict(lr=0.002),
    use_kan=True,
    kan_grid_size=5,
    kan_spline_order=3,
    seed=seed,
    verbose=50
)

In [7]:
# Hyperparameters from original paper.
paper_fit_config = {
    'batch_size': 16384,
    'virtual_batch_size': 512,
}

clf_kan.fit(
    X_train=X_train, y_train=y_train,
    eval_set=[(X_valid, y_valid)],
    eval_name=['valid'],
    eval_metric=['accuracy'],
    **paper_fit_config,
    max_epochs=MAX_EPOCHS,
    patience=100,
    num_workers=0,
    drop_last=False,
    compute_importance=False
)

epoch 0  | loss: 1.69649 | valid_accuracy: 0.40128 |  0:00:09s
epoch 50 | loss: 0.34344 | valid_accuracy: 0.8668  |  0:06:52s
epoch 100| loss: 0.27183 | valid_accuracy: 0.89217 |  0:13:36s
epoch 150| loss: 0.20932 | valid_accuracy: 0.92301 |  0:20:19s
epoch 200| loss: 0.19252 | valid_accuracy: 0.93138 |  0:27:02s
epoch 250| loss: 0.16566 | valid_accuracy: 0.93752 |  0:33:44s
epoch 300| loss: 0.17289 | valid_accuracy: 0.93467 |  0:40:27s
epoch 350| loss: 0.15352 | valid_accuracy: 0.94441 |  0:47:09s
epoch 400| loss: 0.14808 | valid_accuracy: 0.94417 |  0:53:54s
epoch 450| loss: 0.14252 | valid_accuracy: 0.94625 |  1:00:39s
epoch 500| loss: 0.13777 | valid_accuracy: 0.94971 |  1:07:21s
epoch 550| loss: 0.13442 | valid_accuracy: 0.94967 |  1:14:01s
epoch 600| loss: 0.13087 | valid_accuracy: 0.95033 |  1:20:43s
epoch 650| loss: 0.12753 | valid_accuracy: 0.95219 |  1:27:34s
epoch 700| loss: 0.12534 | valid_accuracy: 0.95198 |  1:34:23s
epoch 750| loss: 0.12448 | valid_accuracy: 0.95375 |  1

In [8]:
# sum up all learnable weights in the PyTorch network
param_count = sum(p.numel() for p in clf_kan.network.parameters() if p.requires_grad)

print(f"Total Learnable Parameters: {param_count:,}")

Total Learnable Parameters: 131,046


### Test Evaluation

In [9]:
from sklearn.metrics import accuracy_score

# evaluate on the hold-out test set
preds = clf_kan.predict(X_test)
test_acc = accuracy_score(y_test, preds)

print(f"Test Accuracy: {test_acc:.5f}")

Test Accuracy: 0.95849


### Data Export

In [10]:
BASE_DIR = './kan-tabnet-experiments'
TABLES_DIR = f'{BASE_DIR}/results/forest_cover/tables'
MODELS_DIR = f'{BASE_DIR}/results/forest_cover/models'

os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# package all metrics into a single JSON payload
experiment_results = {
    "model_type": "KAN-TabNet",
    "dataset": "Forest Cover",
    "parameters": param_count,
    "scheduler": "StepLR",
    "final_test_accuracy": test_acc,
    "best_epoch": clf_kan.best_epoch,
    "training_history": clf_kan.history.history
}

# save JSON metrics
JSON_FILEPATH = f'{TABLES_DIR}/07_kan_sensitivity_10x10_metrics.json'
with open(JSON_FILEPATH, 'w') as f:
    json.dump(experiment_results, f, indent=4)
print(f"Metrics successfully saved to {JSON_FILEPATH}")

# save the model weights
_ = clf_kan.save_model(f'{MODELS_DIR}/07_kan_sensitivity_10x10_{param_count // 1000}k')

Metrics successfully saved to ./kan-tabnet-experiments/results/forest_cover/tables/07_kan_sensitivity_10x10_metrics.json
Successfully saved model at ./kan-tabnet-experiments/results/forest_cover/models/07_kan_sensitivity_10x10_131k.zip
